# Refinamento — Ações do Secretário (SEMARH-PI)

Transforma as planilhas **brutas** da agenda do Secretário num dataset **pronto para o painel**.

| | |
|---|---|
| **Origem** | `s3a://entrada/sermarh_painel/dados_tabela_acoes_sec{+,-}AgPoliti.parquet` (as ações) · `municipios_aderentes_atendidos_.parquet` (território, coordenada, população, pacto) · `filtro_acoes_sec{+,-}AgPolitica.parquet` (só para conferência) |
| **Destino** | `nessie.refinamento.semarh_acoes_secretario` (ações) · `..._municipios` (os 224 municípios) |
| **Grão** | ações: 1 linha por ação (data × município) · municípios: 1 linha por município |

**As duas planilhas de ações são a mesma lista, com e sem agenda política.** A que tem o
sufixo `+AgPoliti` traz tudo; a `-AgPoliti` é a mesma lista sem as ações de agenda política.
Em vez de publicar duas tabelas quase idênticas, o refinamento publica **uma** e marca a
coluna `agenda_politica` nas linhas que só existem na versão `+` — assim o painel liga e
desliga esse recorte com um filtro, e a contagem de cidades visitadas acompanha.

A **cidade visitada** não vem de coluna nenhuma: é município com pelo menos uma ação. Os
arquivos `filtro_acoes_sec*` já trazem essa conta pronta (e a data da primeira visita), e é
exatamente contra eles que a seção de validação confere o que foi derivado aqui — se a origem
mudar de forma incoerente, a carga falha em vez de publicar número errado.

Duas normalizações que o refinamento aplica sobre a fonte:

- **Chave de junção sem acento e em caixa alta.** As três fontes grafam o mesmo município de
  jeitos diferentes (`Barra d'Alcântara` × `Barra D'Alcântara`), e não há código do IBGE na
  planilha de ações. O nome normalizado junta as três; o `cod_ibge` vem da dimensão.
- **Linha sem data ou sem município é descartada.** A planilha vem com o rodapé de uma
  segunda tabela colado embaixo (as colunas `Unnamed: *`), que aparece como linha vazia.

## 1. Ler o dado bruto (camada de entrada, no MinIO)

In [ ]:
from pyspark.sql import functions as F
from lakehouse import sessao, ler_arquivo, gravar, perfil, listar

spark = sessao("refinamento-acoes-secretario")

BASE = "sermarh_painel"
com_agenda = ler_arquivo(spark, f"{BASE}/dados_tabela_acoes_sec+AgPoliti.parquet")
sem_agenda = ler_arquivo(spark, f"{BASE}/dados_tabela_acoes_sec-AgPoliti.parquet")
dim_bruta = ler_arquivo(spark, f"{BASE}/municipios_aderentes_atendidos_.parquet")
# Conferencia (secao 3): a conta de cidades visitadas ja pronta na origem.
conf_com = ler_arquivo(spark, f"{BASE}/filtro_acoes_sec+AgPolitica.parquet")
conf_sem = ler_arquivo(spark, f"{BASE}/filtro_acoes_sec-AgPolitica.parquet")

print("ações (com agenda política):", com_agenda.count())
print("ações (sem agenda política):", sem_agenda.count())
print("municípios na dimensão:", dim_bruta.count())
com_agenda.printSchema()

## 2. Refinar

Primeiro a **dimensão de municípios** (os 224 do Piauí), que dá território, coordenada,
população e adesão ao pacto. Depois as **ações**, marcadas com `agenda_politica` e ligadas à
dimensão pelo nome normalizado.

In [ ]:
# Acentos fora e caixa alta: e a unica chave que as tres planilhas compartilham
# (nao ha codigo do IBGE na planilha de acoes, e o nome vem grafado de jeitos
# diferentes em cada uma).
ACENTOS = "ÁÀÂÃÄÉÊÈËÍÎÌÏÓÔÕÒÖÚÛÙÜÇáàâãäéêèëíîìïóôõòöúûùüç"
SEM_ACENTO = "AAAAAEEEEIIIIOOOOOUUUUCaaaaaeeeeiiiiooooouuuuc"


def chave(coluna):
    """Nome do municipio normalizado — a chave de juncao entre as fontes."""
    return F.upper(F.translate(F.trim(F.col(coluna)), ACENTOS, SEM_ACENTO))


# "lat, lon" -> duas colunas numericas (mesma convencao do Selo Ambiental).
coord = F.split(F.regexp_replace(F.col("coord"), " ", ""), ",")

# A dimensao vem com 226 linhas: dois municipios aparecem duas vezes, com o nome
# curto e o longo ("Nazária" e "Nazária do Piauí"), sempre com o MESMO codigo.
# O nome canonico e o da planilha de filtro, que e a que o painel exibe.
nomes = conf_com.select(
    F.col("cod").cast("int").alias("cod_ibge"),
    F.trim(F.col("municipio")).alias("municipio"),
)

municipios = (
    dim_bruta.select(
        F.col("cod").cast("int").alias("cod_ibge"),
        F.trim(F.col("regiao")).alias("territorio"),
        F.col("pop").cast("int").alias("populacao"),
        (F.lower(F.trim(F.col("aderente_pactos_pi"))) == "sim").alias("aderente_pacto"),
        coord.getItem(0).cast("double").alias("latitude"),
        coord.getItem(1).cast("double").alias("longitude"),
    )
    .dropDuplicates(["cod_ibge"])
    .join(nomes, "cod_ibge")
    .withColumn("_chave", chave("municipio"))
    .select("cod_ibge", "municipio", "territorio", "populacao", "aderente_pacto",
            "latitude", "longitude", "_chave")
)
print("municípios:", municipios.count())


def acoes_validas(df):
    """Linhas de agenda de fato: com data e com municipio.

    A planilha traz o rodape de uma segunda tabela colado embaixo (as colunas
    `Unnamed: *`), que aparece como linha sem data e sem municipio.
    """
    return (
        df.select(
            F.to_date(F.col("data").cast("timestamp")).alias("data"),
            F.trim(F.col("municipio")).alias("municipio"),
            F.trim(F.col("descricao")).alias("acao"),
        )
        .filter(F.col("data").isNotNull() & (F.coalesce(F.col("municipio"), F.lit("")) != ""))
        .withColumn("_chave", chave("municipio"))
    )


validas_com = acoes_validas(com_agenda)
validas_sem = acoes_validas(sem_agenda)

# Agenda politica = a acao existe na planilha "+" e nao existe na "-". A marca sai
# da diferenca entre as duas listas, nao do texto da descricao: e a origem quem
# decide o que conta como agenda politica, e o texto nem sempre diz.
chaves_sem = (
    validas_sem.select("data", "_chave", "acao").distinct()
    .withColumn("_na_lista_sem", F.lit(True))
)

acoes = (
    validas_com.join(chaves_sem, ["data", "_chave", "acao"], "left")
    .withColumn("agenda_politica", F.col("_na_lista_sem").isNull())
    # O nome exibido e sempre o da dimensao (o da planilha de acoes so serve de
    # chave): senao o mesmo municipio apareceria com duas grafias na tabela.
    .drop("_na_lista_sem", "municipio")
    .join(municipios, "_chave", "left")
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .select("data", "ano", "mes", "municipio", "cod_ibge", "territorio", "acao",
            "agenda_politica", "aderente_pacto", "populacao", "latitude", "longitude")
)
print("\nações:", acoes.count(),
      "| de agenda política:", acoes.filter("agenda_politica").count())
acoes.groupBy("ano").pivot("agenda_politica", [False, True]).count().orderBy("ano").show()

In [ ]:
# A dimensao ganha o resumo por municipio: visitado (= tem acao), quantas acoes e
# quando foi a primeira e a ultima. Com e sem agenda politica, porque o painel
# alterna entre os dois recortes e as duas contas mudam junto.
def resumo(df, sufixo):
    return df.groupBy("_chave").agg(
        F.count("*").alias(f"qtd_acoes{sufixo}"),
        F.min("data").alias(f"primeira_visita{sufixo}"),
        F.max("data").alias(f"ultima_visita{sufixo}"),
    )


municipios_ref = (
    municipios
    .join(resumo(validas_com, ""), "_chave", "left")
    .join(resumo(validas_sem, "_sem_agenda_politica"), "_chave", "left")
    .fillna(0, ["qtd_acoes", "qtd_acoes_sem_agenda_politica"])
    .withColumn("visitado", F.col("qtd_acoes") > 0)
    .withColumn("visitado_sem_agenda_politica", F.col("qtd_acoes_sem_agenda_politica") > 0)
    .select("cod_ibge", "municipio", "territorio", "populacao", "aderente_pacto",
            "visitado", "visitado_sem_agenda_politica",
            "qtd_acoes", "qtd_acoes_sem_agenda_politica",
            "primeira_visita", "ultima_visita",
            "primeira_visita_sem_agenda_politica", "ultima_visita_sem_agenda_politica",
            "latitude", "longitude")
)

perfil(municipios_ref)
print("visitados:", municipios_ref.filter("visitado").count(),
      "| sem agenda política:", municipios_ref.filter("visitado_sem_agenda_politica").count())

## 3. Validar antes de gravar

A conferência que importa é contra os arquivos `filtro_acoes_sec*`: eles trazem, prontos da
origem, **quem o secretário visitou** e a **data da primeira visita**. Aqui isso é derivado das
ações — se os dois discordarem, alguma planilha veio incoerente e a carga para.

In [ ]:
def conferir(conf, coluna_visitado, coluna_data):
    """Divergências entre o derivado aqui e a conta pronta na origem.

    Devolve (municípios com visitado diferente, municípios com data diferente).
    A `data` do arquivo de filtro é a da PRIMEIRA visita — foi assim que a
    comparação fechou nas duas versões, com e sem agenda política.
    """
    esperado = conf.select(
        F.col("cod").cast("int").alias("cod_ibge"),
        (F.lower(F.trim(F.col("secretario_visitou"))) == "sim").alias("_visitado"),
        F.to_date(F.col("data").cast("timestamp")).alias("_primeira"),
    )
    junto = municipios_ref.join(esperado, "cod_ibge", "full_outer")
    return (
        junto.filter(F.col(coluna_visitado) != F.col("_visitado")).count(),
        junto.filter(F.col("_visitado") & (F.col(coluna_data) != F.col("_primeira"))).count(),
    )


visita_com, data_com = conferir(conf_com, "visitado", "primeira_visita")
visita_sem, data_sem = conferir(conf_sem, "visitado_sem_agenda_politica",
                                "primeira_visita_sem_agenda_politica")

regras = {
    "224 municípios":          municipios_ref.count() == 224,
    "código IBGE único":       municipios_ref.select("cod_ibge").distinct().count() == 224,
    "município único":         municipios_ref.select("municipio").distinct().count() == 224,
    "toda linha com coord":    municipios_ref.filter(
                                   "latitude IS NULL OR longitude IS NULL").count() == 0,
    "toda ação num município conhecido":
                               acoes.filter("cod_ibge IS NULL").count() == 0,
    "toda ação com data":      acoes.filter("data IS NULL").count() == 0,
    "visitados batem com a origem":            visita_com == 0,
    "visitados (sem agenda política) batem":   visita_sem == 0,
    "primeira visita bate com a origem":       data_com == 0 and data_sem == 0,
}
for regra, passou in regras.items():
    print(f"  {'OK  ' if passou else 'FALHOU'}  {regra}")

# Coerencia entre a marca e o texto: nao vira assert (quem manda e a origem),
# mas o que destoar tem de aparecer aqui.
marcadas = acoes.filter("agenda_politica")
sem_texto = marcadas.filter(~F.lower(F.col("acao")).contains("agenda politi")
                            & ~F.lower(F.col("acao")).contains("agenda polít")).count()
print(f"\n  ações marcadas como agenda política sem o termo na descrição: {sem_texto}")
marcadas.select("data", "municipio", "acao").orderBy("data").show(5, truncate=60)

assert all(regras.values()), "corrija antes de gravar"

## 4. Gravar em `refinamento`

In [ ]:
gravar(acoes,          "refinamento.semarh_acoes_secretario",            modo="substituir")
gravar(municipios_ref, "refinamento.semarh_acoes_secretario_municipios", modo="substituir")

## 5. Conferir o resultado

In [ ]:
listar(spark, "refinamento")

print("\nAções por território:")
(acoes.groupBy("territorio").agg(F.count("*").alias("acoes"),
                                 F.countDistinct("municipio").alias("municipios"))
      .orderBy(F.desc("acoes")).show(truncate=False))

print("Municípios mais visitados:")
(municipios_ref.select("municipio", "territorio", "qtd_acoes", "ultima_visita")
               .orderBy(F.desc("qtd_acoes")).show(10, truncate=False))

print("Últimas ações registradas:")
acoes.select("data", "municipio", "territorio", "acao", "agenda_politica") \
     .orderBy(F.desc("data")).show(10, truncate=50)